# 0.24 — Quantum detection across **multiple ETF dates** (genAI-style, per date)

Run the entity-emergence detector in a **5-month window before each quantum investability date**,
each with a **fixed baseline before that window** (short window ⇒ fixed baseline ≈ point-in-time,
exactly like the genAI/CHAT run). One date per row instead of genAI's single CHAT date.

| date | 5-mo detection | baseline (24 mo before) | testable? |
|---|---|---|---|
| BQTUM index 2015-12-18 | Jul–Dec 2015 | 2013–2015 | yes |
| QTUM ETF 2018-09-04 | Apr–Sep 2018 | 2016–2018 | yes |
| CHPX ETF 2025-01-15 | Aug 2024–Jan 2025 | 2022–2024 | yes |
| QTUP ETF 2026-09-01 | Apr–Sep 2026 | 2024–2026 | **no** — no 2026 data (future) |

In [ ]:
import os, re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"
ENV_PATH = _ROOT / ".env"                       # OPENAI_API_KEY etc. for the llm_kept column
if ENV_PATH.exists():
    for _l in ENV_PATH.read_text().splitlines():
        _l = _l.strip()
        if "=" in _l and not _l.startswith("#"):
            _k, _, _v = _l.partition("="); os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

# --- the dates to test ---
DATES = {"BQTUM_index": "2015-12-18", "QTUM_etf": "2018-09-04",
         "CHPX_etf": "2025-01-15", "QTUP_etf": "2026-09-01"}
DETECT_MONTHS = 5          # detection window length, before each date
BASELINE_MONTHS = 24       # fixed baseline length, immediately before the detection window
DATA_LAST_YEAR = 2025      # we have raw/terms only through 2025

# --- detector params (same as the entity-emergence pipeline) ---
FREQ="W-MON"; REP_HL=8; MIN_MENTIONS=3; DEGREE_MIN=8; PERSIST_WEEKS=2; CLUSTER_K=15; CLUSTER_MAX=0.60
TERM_STOP = set(ENGLISH_STOP_WORDS) | {"says","said","new","year","week","report","shares","stock",
    "jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"}
EVENT_STOP = {"collapse","rout","bankruptcy","bankrupt","fraud","lawsuit","sue","sues","sued","probe","hearing",
 "trial","court","arrest","arrested","resign","resigns","ban","bans","banned","outage","recall","default","slump",
 "slumps","plunge","plunges","crash","derailment","strike","quake","earthquake","protest","protests","unrest",
 "attack","war","sanctions","fine","fined","scandal","layoffs","layoff","pivot","halt","halts","delay","delays",
 "death","dies","killed","guilty","charges","charged","indicted","crisis","takeover","merger","deal","acquires",
 "acquire","buys","stake","ipo","listing","bond","bonds","notes","debt","offering","case","settlement","shortage",
 "shutdown","tumbles","soars","jumps","rises","falls","drops","gains","cuts","raises"}
def is_entity(t): return not any(tok in EVENT_STOP for tok in t.split())   # Stage 0 = generic blocklist
HINT = re.compile(r"\bionq\b|rigetti|\bd-wave\b|\bdwave\b|quantinuum|\bqubit|quantum comput|psiquantum|quantum advantage", re.I)
def normalize_terms(v): return v.tolist() if isinstance(v, np.ndarray) else (list(v) if isinstance(v,(list,tuple)) else [])
def filter_terms(ts): return [t for t in ts if len(t)>=3 and not any(tok in TERM_STOP for tok in t.split())]
def week_ts(w): return pd.Period(w, freq=FREQ).start_time
def clustcoef(P, adj):
    if len(P)<2: return 0.0
    tot=pairs=0
    for i in range(len(P)):
        for j in range(i+1,len(P)):
            tot+=1
            if P[j] in adj[P[i]]: pairs+=1
    return pairs/tot if tot else 0.0
print("config ok ·", DETECT_MONTHS, "mo detection ·", BASELINE_MONTHS, "mo baseline · dates:", list(DATES))

## Step 1 — Corpus: build per-year `terms` from raw (self-contained)

**No dependency on pre-existing files.** For any calendar year, if `terms_{year}.parquet` is
missing it is built from `raw_news_{year}.csv.xz` with the extractor below (keep Bloomberg wires →
dedupe → strip source prefixes → unigram+bigram phrases minus finance stopwords). First run builds
+ caches each year; later runs just load. This is the only source of `terms` — nothing else.

In [ ]:
import lzma
import polars as pl
# --- phrase extractor (identical to the code that built every terms_{year}.parquet) ---
WIRES = ["BN", "BFW", "BBO"]
TOKEN_RE  = re.compile(r"(?u)\b[a-z][a-z0-9\-]{2,}\b")
BIGRAM_RE = re.compile(r"(?u)\b[a-z][a-z0-9\-]{2,}\s+[a-z][a-z0-9\-]{2,}\b")
EXTRACT_STOP = set(ENGLISH_STOP_WORDS) | {
    "inc","plc","ltd","llc","corp","co","sa","ag","nv","group","holdings","ceo","cfo","says","said","new","year",
    "today","week","day","update","report","reports","results","announces","announced","shares","stock","stocks",
    "stake","dividend","q1","q2","q3","q4","fy","unit","mln","bln","pct","jan","feb","mar","apr","may","jun","jul",
    "aug","sep","oct","nov","dec","sales","deal","chief","cut","raised","raise","buy","sell","net","revenue","beat",
    "miss","forecast","outlook","business","market","markets","global","first","second","third","fourth","annual",
    "meeting","plans","plan","bn","march","april","june","july","august","september","october","november","december",
    "rated","hold","neutral","perform","outperform","underperform","overweight","underweight","equal-weight","lowers",
    "upgrade","downgrade","maintains","reiterates","est","eps","adj","sees","expects","names","appoints","hires",
    "officer","director","chairman","executive","promotes","tender","offering","offer","notes","bond","bonds","debt","bills","yield"}
def _strip_prefix(text):
    for _ in range(2):
        if ":" not in text: return text
        pre, _, rest = text.partition(":")
        if not pre or not rest or len(pre) > 30 or len(pre.split()) > 4: return text
        text = rest.strip()
    return text
def _extract(text):
    t = text.lower(); terms = set()
    for bg in BIGRAM_RE.findall(t):
        if all(tok not in EXTRACT_STOP for tok in bg.split()): terms.add(bg)
    for tok in TOKEN_RE.findall(t):
        if tok not in EXTRACT_STOP and len(tok) >= 3: terms.add(tok)
    return sorted(terms)

def build_year_terms(yr):
    """Build terms_{yr}.parquet from raw_news_{yr}.csv.xz if not already cached; return its path."""
    out = OUTPUT_DIR / f"terms_{yr}.parquet"
    if out.exists(): return out
    raw = _ROOT / "data" / "raw" / f"raw_news_{yr}.csv.xz"
    if not raw.exists(): raise FileNotFoundError(f"no raw news for {yr}: {raw}")
    with lzma.open(raw, "rb") as f:
        d = (pl.scan_csv(f, infer_schema_length=10_000)
             .select(["Headline", "CaptureTime", "WireName"])
             .filter(pl.col("WireName").is_in(WIRES) & pl.col("Headline").is_not_null())
             .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
             .collect()).to_pandas()
    d["date"] = pd.to_datetime(d["CaptureTime"]).dt.tz_localize(None)
    d = d[d.date.dt.year == yr].dropna(subset=["Headline"]).drop_duplicates("Headline")
    d["Headline"] = d["Headline"].map(_strip_prefix)
    d = d[d.Headline.str.split().map(len) >= 4].reset_index(drop=True)
    d["terms"] = [_extract(h) for h in d["Headline"]]
    d[["Headline", "date", "terms"]].to_parquet(out, index=False)
    print(f"  built terms_{yr}.parquet ({len(d):,} headlines)")
    return out

def load_corpus(y0, y1):
    """Full [Headline, date, terms] for calendar years [y0, y1] — building any missing year from raw."""
    parts = []
    for yr in range(y0, y1 + 1):
        build_year_terms(yr)
        parts.append(pd.read_parquet(OUTPUT_DIR / f"terms_{yr}.parquet")[["Headline", "date", "terms"]])
    df = pd.concat(parts, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    return df
print("self-contained corpus loader ready (builds terms_{year}.parquet from raw on demand)")

## Step 2 — The detector (one row per week × emerging entity)

For each week in the window the detector lists every **novel** entity (≥3 mentions, not an event word),
its **partner words** that week, and the per-week measures: `mentions`, `degree` (how many other words it
co-appears with), `clustering` (1.0 = one tight story, →0 = a bridge across many stories).

In [ ]:
def detect(news, baseline_end, ds, de):
    df = news.copy()
    df["terms_f"] = df["terms"].map(normalize_terms).map(filter_terms)
    df["week"] = df["date"].dt.to_period(FREQ).astype(str)
    baseline_terms = set()
    for tl in df.loc[df.date <= baseline_end, "terms_f"]: baseline_terms.update(tl)   # fixed baseline
    disc = [w for w in sorted(df["week"].unique(), key=week_ts) if ds <= week_ts(w) <= de]
    rows = []
    for week in disc:
        grp = df.loc[df.week == week]
        twc, adj = Counter(), defaultdict(Counter)
        for tl in grp["terms_f"]:
            s = set(tl)
            for t in s: twc[t] += 1
            for a, b in combinations(sorted(s), 2): adj[a][b]+=1; adj[b][a]+=1
        for t, cnt in twc.items():
            if t in baseline_terms or cnt < MIN_MENTIONS or not is_entity(t): continue   # novel + support + entity
            epart = [p for p,_ in adj[t].most_common() if is_entity(p)]; P = epart[:CLUSTER_K]
            reps = [hl for hl,tl in zip(grp["Headline"],grp["terms_f"]) if t in set(tl)][:REP_HL]
            rows.append({"week":week, "anchor":t, "partners":", ".join(epart[:8]),
                         "mentions":int(cnt), "degree":len(epart), "clustering":round(clustcoef(P,adj),3),
                         "subgraph":[t]+P[:10], "reps":reps, "example":reps[0] if reps else ""})
    return pd.DataFrame(rows)

def promotion_week(sub):
    """First week an anchor graduates: caught >=PERSIST_WEEKS, degree at a new high, median clustering a bridge."""
    sub = sub.sort_values("week", key=lambda s: s.map(week_ts))
    wk, deg, clu = list(sub.week), list(sub.degree), list(sub.clustering)
    for i in range(len(wk)):
        if (i+1) >= PERSIST_WEEKS and deg[i] >= max(deg[:i] or [0]) and float(np.median(clu[:i+1])) <= CLUSTER_MAX:
            return wk[i]
    return None
print("detect + promotion_week ready")

## Step 3 — LLM filter (fills the `llm_kept` column)

In the pipeline the LLM only ever judges **promoted** entities — it's a conservative gate that *removes*
clusters that are clearly a single-company event / a macro aggregate / wire boilerplate, and keeps anything
that looks like a real multi-actor technology theme. So `llm_kept` is only defined for promoted rows.

In [ ]:
from pydantic import BaseModel
from typing import Literal
REJECT_SYS = ("You are a conservative FILTER that REMOVES clusters of news headlines that are NOT emerging themes. "
 "You never decide what IS a theme; only flag clusters that CLEARLY are: 1. single_entity_event, "
 "2. macro_market_aggregate, 3. boilerplate_wire. If unclear, KEEP. KEEP anything describing a SPECIFIC "
 "technological/industrial/product development across multiple actors.")
class Reject(BaseModel):
    verdict: Literal["keep","reject"]; category: Literal["single_entity_event","macro_market_aggregate","boilerplate_wire","none"]; reason: str
_client = None
def llm_keep(subgraph, headlines):
    """Return True (keep) / False (reject); None if no API key configured."""
    global _client
    if not os.environ.get("OPENAI_API_KEY"): return None
    from openai import OpenAI
    if _client is None: _client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ.get("OPENAI_BASE") or None)
    user = "Cluster entities: " + ", ".join(subgraph[:14]) + "\nHeadlines:\n" + "\n".join(f"- {h}" for h in headlines[:8]) + "\n\nClassify this cluster."
    r = _client.beta.chat.completions.parse(model=os.environ.get("OPENAI_DEFAULT_MODEL","gpt-4o-mini"), temperature=0,
        response_format=Reject, messages=[{"role":"system","content":REJECT_SYS},{"role":"user","content":user}])
    return r.choices[0].message.parsed.verdict == "keep"
print("llm_keep ready ·", "API key found" if os.environ.get("OPENAI_API_KEY") else "NO API key (llm_kept stays blank)")

## Step 4 — One dataframe per target date

`DFS[date]` is a tidy table: **one row per (week, emerging entity)**, sorted by week then degree.
Columns — `partners` (the other words), then the measures: numeric `mentions / degree / clustering`,
binary `is_quantum / caught / promoted / llm_kept`, plus an `example` headline. Each is saved to
`quantum_weekly_<date>.{parquet,csv}`. The LLM runs only on promoted rows (capped at `LLM_MAX`).

In [ ]:
LLM_MAX = 25
COLS = ["week","anchor","partners","mentions","degree","clustering","is_quantum","caught","promoted","llm_kept","example"]
DFS, summary = {}, []
for name, ds_str in DATES.items():
    D = pd.Timestamp(ds_str)
    det_start  = (D - pd.DateOffset(months=DETECT_MONTHS)).normalize()
    base_start = (det_start - pd.DateOffset(months=BASELINE_MONTHS)).normalize()
    if D.year > DATA_LAST_YEAR:                                   # QTUP — no data yet
        DFS[name] = pd.DataFrame(columns=COLS)
        summary.append({"date":name,"when":D.date(),"rows":0,"caught":0,"promoted":0,"quantum_rows":0,"status":"no data (future)"})
        print(f"{name} ({D.date()}): SKIP — needs {D.year} data we don't have"); continue

    corpus = load_corpus(base_start.year, D.year)
    corpus = corpus[(corpus.date >= base_start) & (corpus.date <= D)]
    R = detect(corpus, baseline_end=det_start, ds=det_start, de=D)

    # --- per-row measures ---
    R["is_quantum"] = R["anchor"].str.contains(HINT, na=False)
    R["caught"]     = (R.mentions >= MIN_MENTIONS) & (R.degree >= DEGREE_MIN)
    R["promoted"]   = False
    for a, sub in R[R.caught].groupby("anchor"):                  # mark the graduation week per anchor
        pw = promotion_week(sub)
        if pw is not None: R.loc[(R.anchor == a) & (R.week == pw), "promoted"] = True

    # --- llm_kept only on promoted rows (the pipeline's last gate) ---
    R["llm_kept"] = pd.NA
    prom_idx = list(R.index[R.promoted])
    for i in prom_idx[:LLM_MAX]:
        R.at[i, "llm_kept"] = llm_keep(R.at[i,"subgraph"], R.at[i,"reps"])

    out = R.sort_values(["week","degree"], ascending=[True, False])[COLS].reset_index(drop=True)
    DFS[name] = out
    out.to_parquet(OUTPUT_DIR / f"quantum_weekly_{name}.parquet", index=False)
    out.drop(columns=[]).to_csv(OUTPUT_DIR / f"quantum_weekly_{name}.csv", index=False)
    summary.append({"date":name,"when":D.date(),"rows":len(out),"caught":int(out.caught.sum()),
                    "promoted":int(out.promoted.sum()),"quantum_rows":int(out.is_quantum.sum()),"status":"ok"})
    print(f"{name} ({D.date()}): {len(out):,} rows · {int(out.caught.sum()):,} caught · "
          f"{int(out.promoted.sum())} promoted · {int(out.is_quantum.sum())} quantum rows  -> quantum_weekly_{name}.csv")
print("\nSUMMARY"); print(pd.DataFrame(summary).to_string(index=False))

## How an entity gets selected — the funnel

Everything below is computed **per week**, from that week's headlines. An entity must pass four gates in order:

`candidates → caught → promoted → llm_kept`

### 1 · Candidates — *novel, supported, entity-like*

For each week in the window:

- Each headline already carries its `terms` (unigram + bigram phrases, stopwords stripped — extracted once per year and cached). `filter_terms` drops anything under 3 characters or containing a stopword.
- We build the week's **co-occurrence graph**: every pair of terms sharing a headline becomes an edge (`adj`), and each term accumulates a `mentions` count (`twc`).

A term `t` is emitted as a candidate row only if it clears **three entry gates**:

| gate | rule | what it removes |
|---|---|---|
| **novel** | `t ∉ baseline_terms` — never seen before the window (headlines dated ≤ `baseline_end`) | anything already established |
| **support** | `mentions ≥ MIN_MENTIONS` (3) | one-off noise |
| **entity** | `is_entity(t)` — no token in the `EVENT_STOP` blocklist | events: `collapse`, `ipo`, `merger`, … |

Each surviving row stores its measures: `mentions`, `degree`, `clustering`, `partners` (top-8 neighbours) and a few `example` headlines.

### 2 · caught — *connected enough this week*

A per-row test: `caught = (mentions ≥ 3) AND (degree ≥ DEGREE_MIN=8)`. Support is already met, so the binding condition is **`degree ≥ 8`** — the anchor shares a headline with at least 8 distinct other entities this week.

- **`degree`** = number of distinct entity-neighbours of `t`.
- **`clustering`** = among `t`'s top-15 neighbours, the fraction of neighbour-*pairs* that also co-occur with each other. **`≈1.0` = one tight story; `→0` = a hub bridging separate stories.**

### 3 · promoted — *persists, grows, bridges*

Here we shift from per-week to **per-anchor, across weeks**. We sort an anchor's caught weeks in time and promote it at the **first week** where all three signals hold:

| signal | rule | meaning |
|---|---|---|
| **persist** | caught for ≥ `PERSIST_WEEKS` (2) | not a one-week blip |
| **grow** | `degree` at a new high vs. earlier weeks | reach is expanding |
| **bridge** | median `clustering` so far ≤ `CLUSTER_MAX` (0.60) | spans many stories, not a single-event clique |

That week is stamped `promoted = True`. This is the decisive cut — for CHPX, **4,217 caught → 43 promoted**.

### 4 · llm_kept — *survives the reject filter* (final gate)

Run only on the top `LLM_MAX` (25) promoted clusters per date:

- **Input** — the cluster's entities (`subgraph` = anchor + top partners) plus a handful of `example` headlines.
- **Role** (`REJECT_SYS`) — a deliberately *conservative* filter: reject **only** if the cluster is clearly a (1) single-entity event, (2) macro / market aggregate, or (3) wire boilerplate; **if unsure, keep**; always keep a specific technological / industrial development spanning *multiple actors*.
- **Output** — `verdict ∈ {keep, reject}` → `llm_kept = (verdict == "keep")`. `None` means no API key, or the cluster fell outside the top-25 judged.

**Finally selected = `llm_kept == True`** — i.e. it passed all four gates.

> ⚠️ **This last gate is currently the unreliable one.** It rejected *ChatGPT* at its promotion week despite a textbook emergence (degree 15 → 119 over 20 weeks, ~4 months before the CHAT ETF). So the zero "finally selected" count is a filter problem, **not** a detection failure — gates 1–3 work.


## Step 4b — Position at each funnel step (ranking columns)

Adds, to every `DFS[date]`, where each anchor sits at each gate (lower rank = stronger):

- `rank_candidate` — by `mentions` among **that week's candidates** (step 1)
- `rank_caught` — by `degree` among **that week's caught** anchors (step 2; `<NA>` if not caught)
- `rank_promoted` — by `degree` among the **window's promoted** anchors (step 3; `<NA>` if not promoted). This is the order they're fed to the LLM.
- `llm_judged` — `True` if `rank_promoted ≤ LLM_MAX` (actually sent to the LLM)


In [31]:
# Position at each funnel step — add rank columns to every DFS[date], then re-save.
for name, _df in list(DFS.items()):
    if _df.empty: continue
    df = _df.copy()
    df["rank_candidate"] = df.groupby("week")["mentions"].rank(ascending=False, method="min").astype("Int64")  # step 1
    df["rank_caught"] = pd.Series(pd.NA, index=df.index, dtype="Int64")                                         # step 2
    cr = df[df.caught].groupby("week")["degree"].rank(ascending=False, method="min")
    df.loc[cr.index, "rank_caught"] = cr.astype(int)
    df["rank_promoted"] = pd.Series(pd.NA, index=df.index, dtype="Int64")                                       # step 3
    pm = df[df.promoted].sort_values("degree", ascending=False)
    df.loc[pm.index, "rank_promoted"] = list(range(1, len(pm) + 1))
    df["llm_judged"] = df["rank_promoted"].notna() & (df["rank_promoted"] <= LLM_MAX)                           # step 4 gate
    DFS[name] = df
    df.to_parquet(OUTPUT_DIR / f"quantum_weekly_{name}.parquet", index=False)
    df.to_csv(OUTPUT_DIR / f"quantum_weekly_{name}.csv", index=False)
    print(f"{name}: +rank columns · promoted {int(df.promoted.sum())} · sent to LLM {int(df.llm_judged.sum())}")
_ex = next((d for d in DFS.values() if not d.empty), None)
if _ex is not None: print("\ncolumns now:", list(_ex.columns))

BQTUM_index: +rank columns · promoted 99 · sent to LLM 25
QTUM_etf: +rank columns · promoted 92 · sent to LLM 25
CHPX_etf: +rank columns · promoted 43 · sent to LLM 25

columns now: ['week', 'anchor', 'partners', 'mentions', 'degree', 'clustering', 'is_quantum', 'caught', 'promoted', 'llm_kept', 'example', 'rank_candidate', 'rank_caught', 'rank_promoted', 'llm_judged']


## Step 5 — For each date, each week: the top-K anchors with **all columns**

`WEEKLY[date]` keeps, for every week in that date's window, the `PER_WEEK` highest-`degree` anchors,
with **every** column — `partners` (the other words), the numeric measures (`mentions`, `degree`,
`clustering`), the binary flags (`is_quantum`, `caught`, `promoted`, `llm_kept`), and an `example`
headline. Each date renders as its own table; the full untrimmed table is still `DFS[date]` / the CSV.

In [32]:
from IPython.display import display
PER_WEEK = 10        # top anchors to show FOR EACH WEEK (ranked by degree)
ALLCOLS = ["week","anchor","partners","mentions","degree","clustering","is_quantum","caught","promoted","llm_kept","example"]
pd.set_option("display.max_colwidth", 55); pd.set_option("display.max_rows", 800); pd.set_option("display.width", 240)
WEEKLY = {}
for name in DATES:
    df = DFS[name]
    if df.empty:
        print(f"### {name}: no data (future date)"); continue
    wk = (df.sort_values(["week","degree"], ascending=[True, False])
            .groupby("week", sort=True).head(PER_WEEK).reset_index(drop=True))
    WEEKLY[name] = wk
    wk[ALLCOLS].to_csv(OUTPUT_DIR / f"quantum_weekly_top{PER_WEEK}_{name}.csv", index=False)
    print(f"### {name} — top {PER_WEEK} anchors PER WEEK · {wk.week.nunique()} weeks · "
          f"{int(df.is_quantum.sum())} quantum row(s) in full table")
    qr = df[df.is_quantum].sort_values("week")          # PINNED: always show quantum, even if below the top-K
    if len(qr):
        print("  ↓ quantum rows (pinned — shown regardless of degree rank):")
        display(qr[ALLCOLS])
    display(wk[ALLCOLS])

### BQTUM_index — top 10 anchors PER WEEK · 22 weeks · 0 quantum row(s) in full table


,week,anchor,partners,mentions,degree,clustering,is_quantum,caught,promoted,llm_kept,example
0,2015-07-21/2015-07-27,dnya,"cebus, danya, danya cebus, ils, loss, shekels, inco...",27,70,0.619,False,True,False,<NA>,*DANYA CEBUS SAYS CONTRACT WORTH ABOUT 600 MLN SHEK...
1,2015-07-21/2015-07-27,american colony,"american, colony, israel, project, amcl, sale, chan...",15,52,0.629,False,True,False,<NA>,*AMERICAN COLONY TO TAKE 61M ILS 2Q CHARGE FOR ASSE...
2,2015-07-21/2015-07-27,dongwon financial,"dongwon, financial, korea, earnings, table, south, ...",15,52,0.638,False,True,False,<NA>,Dongwon Financial Holdings Shares Drop in South Kor...
3,2015-07-21/2015-07-27,technivie,"abbvie, approval, fda, gets, chronic, hepatitis, ch...",15,46,0.590,False,True,False,<NA>,TECHNIVIE & VIEKIRA PARK ARE MARKETED BY ABBVIE
4,2015-07-21/2015-07-27,austrac,"tabcorp, proceedings, matters, tah, brought, civil,...",11,38,0.410,False,True,False,<NA>,AUSTRAC PROCEEDINGS AGAINST TABCORP
5,2015-07-21/2015-07-27,tsavo,"media, cyberplex, tsavo media, committee, pact, alt...",12,37,0.533,False,True,False,<NA>,*TSAVO MEDIA SAYS COMMITTEE FORMED TO REVIEW ALTERN...
6,2015-07-21/2015-07-27,conpharm,"table, loss, sk0, loss sk0, share, pretax, shr, sk1",20,36,0.648,False,True,False,<NA>,"*CONPHARM 1H OP PROFIT SK722,000 VS SK947,000 ..."
7,2015-07-21/2015-07-27,gavis,"lupin, expand, generic, products, generics, million...",17,36,0.229,False,True,False,<NA>,*LUPIN ACQUIRES GAVIS TO EXPAND US GENERIC BUSINESS
8,2015-07-21/2015-07-27,wah shing,"shing, wah, declares, holding, international, intl,...",8,35,0.562,False,True,False,<NA>,"Wah Shing's Hui Discusses Strategy, Toy Market & China"
9,2015-07-21/2015-07-27,burst media,"burst, media, cyberplex, approach, acquisition, cor...",7,35,0.590,False,True,False,<NA>,*CYBERPLEX INC. PROPOSES ACQUISITION OF BURST MEDIA...


### QTUM_etf — top 10 anchors PER WEEK · 22 weeks · 0 quantum row(s) in full table


,week,anchor,partners,mentions,degree,clustering,is_quantum,caught,promoted,llm_kept,example
0,2018-04-10/2018-04-16,dabbab,"abu, gippsland, abu dabbab, project, gip, alluvial,...",31,103,0.705,False,True,False,<NA>,Gippsland Agrees Tantalum Sale From Abu Dabbab Proj...
1,2018-04-10/2018-04-16,cofetel,"mexico, telecom, amxl, statement, connection, link,...",22,90,0.648,False,True,False,<NA>,Mexico’s Cofetel to Unveil Connection Cost Model: N...
2,2018-04-10/2018-04-16,abu dabbab,"abu, dabbab, gippsland, project, gip, files, gippsl...",22,72,0.676,False,True,False,<NA>,*GIPPSLAND TO MAKE ANNOUNCEMENT ON ABU DABBAB PROJE...
3,2018-04-10/2018-04-16,plh,"plymouth, minerals, plymouth minerals, zambia, cash...",16,52,0.505,False,True,False,<NA>,*PLH PLYMOUTH TO FOCUS ON TUNGSTEN-TIN
4,2018-04-10/2018-04-16,sentiment mixed,"mixed, sentiment, amid, data, ahead, economic, econ...",15,49,0.524,False,True,False,<NA>,Sentiment Mixed Ahead of Debt Limit Vote
5,2018-04-10/2018-04-16,fury-1,"monitor, oil, testing, energy, monitor energy, petr...",12,47,0.571,False,True,False,<NA>,COE:FURY-1 INITIAL FLOW RATES REACH 75 BARRELS OF O...
6,2018-04-10/2018-04-16,syria risk,"risk, syria, inside, auction, axes, ruble, russia, ...",6,42,0.667,False,True,False,<NA>,"Russia Axes Bond Auction as Sanctions, Syria Risk B..."
7,2018-04-10/2018-04-16,platinum futures,"futures, platinum, tokyo, yen, gram, limit, daily, ...",35,39,0.867,False,True,False,<NA>,*PLATINUM FUTURES IN TOKYO DECLINE BY 300-YEN DAILY...
8,2018-04-10/2018-04-16,gippsland files,"files, gippsland, gip, project, tin, abu, abu dabba...",13,39,0.648,False,True,False,<NA>,*GIPPSLAND FILES ADOBHA DRILLING RESULTS ...
9,2018-04-10/2018-04-16,soybeans called,"called, soybeans, lower, demand, wheat, grains, hig...",12,38,0.848,False,True,False,<NA>,"Grain, Soybeans Called Lower as Japan Quake May Lim..."


### CHPX_etf — top 10 anchors PER WEEK · 22 weeks · 1 quantum row(s) in full table
  ↓ quantum rows (pinned — shown regardless of degree rank):


,week,anchor,partners,mentions,degree,clustering,is_quantum,caught,promoted,llm_kept,example
5408,2025-01-07/2025-01-13,ionq delta,"daily, daily volume, delta, hedge, ionq, options, o...",3,9,1.0,True,True,False,<NA>,IonQ Delta Hedge at 12% Daily Volume: Options Pre-Mkt


,week,anchor,partners,mentions,degree,clustering,is_quantum,caught,promoted,llm_kept,example
0,2024-08-20/2024-08-26,wukong,"game, tencent-backed, first-day, history, makes, ch...",9,54,0.562,False,True,False,<NA>,Tencent-Backed ‘Wukong’ Makes Gaming History in Fir...
1,2024-08-20/2024-08-26,durov,"telegram, france, afp, airport, paris, paris airpor...",8,35,0.362,False,True,False,<NA>,"Telegram CEO Pavel Durov Arrested at Paris Airport,..."
2,2024-08-20/2024-08-26,lynch yacht,"lynch, yacht, bloomer, missing, morgan, morgan stan...",6,34,0.610,False,True,False,<NA>,"Morgan Stanley’s Bloomer Among Lynch Yacht Missing,..."
3,2024-08-20/2024-08-26,freixe,"nestle, laurent, schneider, mark, laurent freixe, s...",12,32,0.486,False,True,False,<NA>,*NESTLÉ NAMES LAURENT FREIXE AS CEO
4,2024-08-20/2024-08-26,v-wave,"johnson, payment, upfront, medtech, diluting, upfro...",13,30,0.162,False,True,False,<NA>,Johnson & Johnson to Buy V-Wave
5,2024-08-20/2024-08-26,canada railways,"canada, railways, shippers, shut, lock, snarling, s...",6,28,0.505,False,True,False,<NA>,Trudeau Urges a Deal as Canada Railways Edge Closer...
6,2024-08-20/2024-08-26,ww2 bomb,"bomb, ww2, czech, discovery, follow, litvinov, move...",3,23,0.657,False,True,False,<NA>,*ORLEN LITVINOV MOVES FOLLOW DISCOVERY OF WW2 BOMB ...
7,2024-08-20/2024-08-26,italy coast,"coast, guard, italy, lynch, yacht, bloomer, rescue,...",3,21,0.714,False,True,False,<NA>,"*ITALY COAST GUARD SAYS UNLIKELY TO RESCUE LYNCH, B..."
8,2024-08-20/2024-08-26,altrad,"beerenberg, cash, nok41, recommended, beerenberg bo...",4,21,0.457,False,True,False,<NA>,*BEERENBERG BOARD UNANIMOUSLY RECOMMENDS OFFER BY A...
9,2024-08-20/2024-08-26,wamco,"trades, king, longtime, shunned, spotlight, thrust,...",8,21,0.267,False,True,False,<NA>,*US CRIMINAL PROBE INTO WAMCO SAID TO FOCUS ON 'CHE...


### QTUP_etf: no data (future date)


In [33]:
DFS['CHPX_etf'][DFS['CHPX_etf'].is_quantum]

,week,anchor,partners,mentions,degree,clustering,is_quantum,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
5408,2025-01-07/2025-01-13,ionq delta,"daily, daily volume, delta, hedge, ionq, options, o...",3,9,1.0,True,True,False,<NA>,IonQ Delta Hedge at 12% Daily Volume: Options Pre-Mkt,139,205,<NA>,False
